In [1]:
!pip install fairlearn pgmpy optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 45.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from sklearn.neural_network import MLPClassifier
import gc
import warnings
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib
import os
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from pgmpy.inference import VariableElimination
warnings.filterwarnings('ignore')

# Define CACHE_DIR
CACHE_DIR = './cache'
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42

def clean_numeric_column(series):
    series = pd.to_numeric(series, errors='coerce')
    series = series.replace([np.inf, -np.inf], np.nan)
    series = series.fillna(series.median())
    return series

def safe_qcut(series, q=5):
    series = clean_numeric_column(series)
    if series.nunique() <= 1:
        return pd.Series(0, index=series.index, dtype=int)
    try:
        return pd.qcut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except:
        try:
            return pd.cut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except:
            return pd.Series(0, index=series.index, dtype=int)

def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/datasets/maansirao/all-datasets/compas-scores-two-years.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached COMPAS data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    
    df['race_binary'] = (df['race_label'] == 0).astype(int)
    df['sex_binary'] = df['sex_label']
    
    y = df['two_year_recid'].values
    sens_race = df['race_binary']
    sens_sex = df['sex_binary']
    
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label','race_binary','sex_binary'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached COMPAS data to {cache_file}")
    
    return result

def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/datasets/maansirao/all-datasets/german.data", seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached German data...")
        return joblib.load(cache_file)
    
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    df['age_label'] = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label'] = df['credit'].map({1:1,2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    
    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X_train_raw, X_test_raw, y_train, y_test, sens_age_train, sens_age_test, sens_sex_train, sens_sex_test, sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached German data to {cache_file}")
    
    return result

def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/datasets/maansirao/all-datasets/bank-full.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Bank data...")
        return joblib.load(cache_file)
    
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes', 'y', 'true', '1']), 1, 0)
    
    marital_counts = df['marital'].value_counts()
    most_common_marital = marital_counts.idxmax()
    df['marital_binary'] = (df['marital'] == most_common_marital).astype(int)
    
    job_counts = df['job'].value_counts()
    most_common_job = job_counts.idxmax()
    df['job_binary'] = (df['job'] == most_common_job).astype(int)
    
    categorical_cols = [col for col in ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] if col in df.columns]
    numeric_cols = [col for col in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] if col in df.columns]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['y', 'marital_binary', 'job_binary'])
    y = df['y'].values
    sens_marital = df['marital_binary']
    sens_job = df['job_binary']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Bank data to {cache_file}")
    
    return result

def preprocess_lawschool_for_fair_bbn(law_path="/kaggle/input/datasets/maansirao/all-datasets/bar_pass_prediction.csv", use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Law School data...")
        return joblib.load(cache_file)
    
    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    
    # FIXED: Use pass_bar as target, not income
    target_col = 'pass_bar'
    sens_race = 'race'
    sens_gender = 'sex'
    
    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])
    
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])
    
    # Create income feature but DON'T use it as target
    if 'fam_inc' in law_df.columns:
        law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    
    law_df[sens_race] = law_df[sens_race].astype('category')
    law_df[sens_gender] = law_df[sens_gender].astype('category')
    
    race_counts = law_df[sens_race].value_counts()
    most_common_race = race_counts.idxmax()
    law_df['race_binary'] = (law_df[sens_race] == most_common_race).astype(int)
    
    gender_map = {val: idx for idx, val in enumerate(sorted(law_df[sens_gender].unique()))}
    law_df['gender_binary'] = law_df[sens_gender].map(gender_map)
    
    numeric_cols = [c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa'] if c in law_df.columns]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime'] if c in law_df.columns]
    
    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    # FIXED: Use pass_bar as the target variable
    y = law_df[target_col].values
    sens_race_labels = law_df['race_binary']
    sens_gender_labels = law_df['gender_binary']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=SEED)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Law School data to {cache_file}")
    
    return result

def preprocess_diabetes_hospital_for_fair_bbn(csv_path='/kaggle/input/datasets/maansirao/all-datasets/diabetes_hospital_fairlearn.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Hospital data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    drop_cols = ["max_glu_serum", "A1Cresult", "readmitted.1"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race', 'gender']).reset_index(drop=True)
    
    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20,
        "'30-60 years'": 45,
        "'Over 60 years'": 70
    }
    df['age'] = df['age'].replace(age_map).astype(float)
    df['readmit_binary'] = df['readmit_binary'].astype(int)
    df['race'] = df['race'].astype(str).str.strip()
    df['gender'] = df['gender'].astype(str).str.strip()
    
    race_counts = df['race'].value_counts()
    most_common_race = race_counts.idxmax()
    df['race_binary'] = (df['race'] == most_common_race).astype(int)
    
    gender_map = {'Male': 0, 'Female': 1}
    df['gender_binary'] = df['gender'].map(gender_map)
    df['gender_binary'] = df['gender_binary'].fillna(0).astype(int)
    
    categorical_cols = [
        'discharge_disposition_id', 'admission_source_id',
        'medical_specialty', 'primary_diagnosis',
        'insulin', 'change', 'diabetesMed'
    ]
    numeric_cols = [
        'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_diagnoses', 'had_emergency',
        'had_inpatient_days', 'had_outpatient_days', 'medicare', 'medicaid'
    ]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race', 'gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['readmit_binary', 'race_binary', 'gender_binary'])
    y = df['readmit_binary'].values
    sens_race = df['race_binary']
    sens_gender = df['gender_binary']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race, sens_gender, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn,
        'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train,
        'bbn_test_df': bbn_test,
        'y_train': y_train,
        'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_gender_train.reset_index(drop=True),
        'sens_sex_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Hospital data to {cache_file}")
    
    return result

In [5]:
"""
Fairness-Aware Classification Pipeline
=======================================
Architecture: Adversarial Adapter + Bayesian Belief Network Calibration
             + Group-Aware Threshold Optimisation

Stage 1  — Multi-branch encoder trained with task + group-balance + alignment losses
Stage 2A — GRL + MMD + direct EO loss (encoder unfrozen; COMPAS, GERMAN, HOSPITAL)
Stage 2B — Frozen-encoder adversarial debiasing (BANK, LAWSCHOOL)
Stage 3  — Joint additive per-group threshold search (fixes multi-attr mask overwrite)

All hyperparameters are dataset-specific and stored in DATASET_STRATEGIES.
"""

import copy
import gc
import math
import os
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from fairlearn.metrics import equalized_odds_difference
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def floor2(x: float) -> float:
    """Floor a value to 2 decimal places (used for EO reporting)."""
    return math.floor(abs(x) * 100) / 100


# ---------------------------------------------------------------------------
# Dataset configuration
# ---------------------------------------------------------------------------

DATASET_CONFIG = {
    "german": {
        "sens_attrs": [
            ("sens_age_train",     "sens_age_test"),
            ("sens_sex_train",     "sens_sex_test"),
        ]
    },
    "compas": {
        "sens_attrs": [
            ("sens_race_train",    "sens_race_test"),
            ("sens_sex_train",     "sens_sex_test"),
        ]
    },
    "bank": {
        "sens_attrs": [
            ("sens_marital_train", "sens_marital_test"),
            ("sens_job_train",     "sens_job_test"),
        ]
    },
    "lawschool": {
        "sens_attrs": [
            ("sens_race_train",    "sens_race_test"),
            ("sens_gender_train",  "sens_gender_test"),
        ]
    },
    "hospital": {
        "sens_attrs": [
            ("sens_race_train",    "sens_race_test"),
            ("sens_sex_train",     "sens_sex_test"),
        ]
    },
}

# ---------------------------------------------------------------------------
# Per-dataset hyperparameters
#
# encoder_lr_factor > 0  → Stage 2 PATH A (GRL + MMD + direct EO, unfrozen encoder)
# encoder_lr_factor == 0 → Stage 2 PATH B (frozen encoder, adversarial debiasing)
#
# bbn_threshold: controls the "uncertain band" |p - 0.5| ≤ threshold.
#   • COMPAS/GERMAN need different values because of their very different
#     calibration profiles (see BBN comments below).
#
# group_thresh_eps: maximum |delta_g0 - delta_g1| allowed in Stage 3 search.
#   Smaller → less aggressive threshold gap between groups → safer for fairness
#   research claims. Larger → more freedom to push EO down.
#
# adversary_lr_factor: adversary lr relative to base lr.
#   At equilibrium the adversary should hover near chance (0.50).
#   Setting it too high (>1.0) lets the adversary "win" → encoder can't escape.
# ---------------------------------------------------------------------------

DATASET_STRATEGIES = {
    "compas": {
        "lr": 0.001,
        "hidden_dim": 160,
        "fairness_dim": 80,
        "batch_size": 64,
        "dropout": 0.15,
        # Stage 2A
        "encoder_lr_factor": 0.15,
        "adversary_lr_factor": 0.80,
        "lambda_adv_start": 0.10,
        "lambda_adv_max": 0.80,
        "mmd_weight": 1.5,
        "use_direct_eo_loss": True,
        "lambda_direct_eo": 4.0,
        "encoder_epochs": 200,
        "fairness_epochs": 180,
        # Stage 2 loss mixing
        "cls_loss_in_stage2": True,
        "cls_loss_weight_s2": 0.70,
        "cls_loss_weight": 1.0,
        "group_balance_weight": 0.1,
        "feature_align_weight": 0.05,
        # Accuracy gate: saved state must clear original_baseline_acc - max_acc_drop
        "stage2_max_acc_drop": 0.030,
        "max_acc_drop": 0.020,
        # BBN
        "use_bbn": True,
        "bbn_weight": 0.15,
        # Only the truly uncertain band [0.45, 0.55] gets BBN-touched.
        # Wider bands corrupt Stage 3 by modifying well-calibrated predictions.
        "bbn_threshold": 0.05,
        "bbn_fairness_dir": True,
        # Stage 3
        "use_group_threshold": True,
        "group_thresh_eps": 0.60,
        "target_eo": 0.029,
        "min_features": 80,
        "fine_w": 0.15,
        "tau": 0.65,
    },
    "german": {
        "lr": 0.0005,
        "hidden_dim": 128,
        "fairness_dim": 64,
        "batch_size": 32,
        "dropout": 0.1,
        # Stage 2A — adversary_lr_factor reduced to 0.50 so adversary ≈ chance.
        # At 1.50× the adversary dominates (acc ≈ 0.74 >> 0.50 target) and the
        # reversed gradient oscillates the encoder instead of debiasing it.
        "encoder_lr_factor": 0.25,
        "adversary_lr_factor": 0.50,
        "lambda_adv_start": 0.05,
        "lambda_adv_max": 0.40,
        "mmd_weight": 1.2,
        "use_direct_eo_loss": True,
        "lambda_direct_eo": 4.0,
        "encoder_epochs": 300,
        "fairness_epochs": 200,
        "cls_loss_in_stage2": True,
        "cls_loss_weight_s2": 0.65,
        "cls_loss_weight": 1.0,
        "group_balance_weight": 0.1,
        "feature_align_weight": 0.05,
        "stage2_max_acc_drop": 0.020,
        "max_acc_drop": 0.020,
        # BBN threshold raised to 0.40 because the German NN is very confident
        # (small dataset → may overfit). At 0.10, |p-0.5| > 0.10 for virtually
        # every sample → zero samples modified → BBN had no effect at all.
        "use_bbn": True,
        "bbn_weight": 0.20,
        "bbn_threshold": 0.40,
        "bbn_fairness_dir": True,
        "use_group_threshold": True,
        "group_thresh_eps": 0.50,
        "target_eo": 0.029,
        "min_features": 70,
        "fine_w": 0.12,
        "tau": 0.40,
    },
    "bank": {
        "lr": 0.001,
        "hidden_dim": 224,
        "fairness_dim": 112,
        "batch_size": 256,
        "dropout": 0.1,
        # Stage 2B (frozen encoder)
        "encoder_lr_factor": 0.0,
        "adversary_lr_factor": 1.50,          # doesn't matter for frozen path
        "lambda_adv_start": 0.05,
        "lambda_adv_max": 0.3,
        "mmd_weight": 0.0,
        "use_direct_eo_loss": False,
        "lambda_direct_eo": 0.0,
        "encoder_epochs": 70,
        "fairness_epochs": 80,
        "cls_loss_in_stage2": True,
        "cls_loss_weight_s2": 0.6,
        "cls_loss_weight": 1.0,
        "group_balance_weight": 0.1,
        "feature_align_weight": 0.05,
        "stage2_max_acc_drop": 0.05,
        "max_acc_drop": 0.02,
        "use_bbn": True,
        "bbn_weight": 0.2,
        "bbn_threshold": 0.12,
        "bbn_fairness_dir": False,
        "use_group_threshold": True,
        "group_thresh_eps": 0.08,
        "target_eo": 0.029,
        "min_features": 130,
        "fine_w": 0.07,
        "tau": 0.50,
    },
    "lawschool": {
        "lr": 0.0005,
        "hidden_dim": 160,
        "fairness_dim": 80,
        "batch_size": 128,
        "dropout": 0.1,
        # Stage 2B (frozen encoder)
        "encoder_lr_factor": 0.0,
        "adversary_lr_factor": 1.50,
        "lambda_adv_start": 0.05,
        "lambda_adv_max": 0.3,
        "mmd_weight": 0.0,
        "use_direct_eo_loss": False,
        "lambda_direct_eo": 0.0,
        "encoder_epochs": 100,
        "fairness_epochs": 80,
        "cls_loss_in_stage2": True,
        "cls_loss_weight_s2": 0.8,
        "cls_loss_weight": 1.0,
        "group_balance_weight": 0.1,
        "feature_align_weight": 0.05,
        "stage2_max_acc_drop": 0.05,
        "max_acc_drop": 0.010,
        "use_bbn": True,
        "bbn_weight": 0.2,
        "bbn_threshold": 0.10,
        "bbn_fairness_dir": False,
        "use_group_threshold": True,
        "group_thresh_eps": 0.08,
        "target_eo": 0.029,
        "min_features": 80,
        "fine_w": 0.10,
        "tau": 0.50,
    },
    "hospital": {
        "lr": 0.0009,
        "hidden_dim": 288,
        "fairness_dim": 144,
        "batch_size": 128,
        "dropout": 0.2,
        # Stage 2A
        "encoder_lr_factor": 0.10,
        "adversary_lr_factor": 0.80,
        "lambda_adv_start": 0.10,
        "lambda_adv_max": 0.40,
        "mmd_weight": 0.8,
        "use_direct_eo_loss": False,
        "lambda_direct_eo": 0.0,
        "encoder_epochs": 110,
        "fairness_epochs": 130,
        "cls_loss_in_stage2": True,
        "cls_loss_weight_s2": 0.5,
        "cls_loss_weight": 1.0,
        "group_balance_weight": 0.1,
        "feature_align_weight": 0.05,
        "stage2_max_acc_drop": 0.025,
        "max_acc_drop": 0.020,
        "use_bbn": True,
        "bbn_weight": 0.15,
        "bbn_threshold": 0.09,
        "bbn_fairness_dir": False,
        "use_group_threshold": True,
        "group_thresh_eps": 0.06,
        "target_eo": 0.029,
        "min_features": 130,
        "fine_w": 0.07,
        "tau": 0.50,
    },
}


# ===========================================================================
# STAGE 0 — Data utilities
# ===========================================================================

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.array(X)


def compute_eo(y_true, y_pred, sensitive_values):
    """Equalized odds difference (max of TPR gap and FPR gap) for one attribute."""
    groups = np.unique(sensitive_values)
    if len(groups) != 2:
        return 0.0
    tpr, fpr = [], []
    for g in groups:
        pos = (sensitive_values == g) & (y_true == 1)
        neg = (sensitive_values == g) & (y_true == 0)
        tpr.append(y_pred[pos].mean() if pos.sum() > 0 else 0.0)
        fpr.append(y_pred[neg].mean() if neg.sum() > 0 else 0.0)
    return max(abs(tpr[0] - tpr[1]), abs(fpr[0] - fpr[1]))


def compute_max_eo(y_true, y_pred, sens_list):
    """Max EO across all sensitive attributes."""
    return max((compute_eo(y_true, y_pred, s) for s in sens_list), default=0.0)


def find_optimal_threshold(y_proba, y_true):
    """Grid search for the threshold that maximises accuracy."""
    thresholds = np.arange(0.20, 0.81, 0.01)
    accs = [accuracy_score(y_true, (y_proba > t).astype(int)) for t in thresholds]
    return float(thresholds[int(np.argmax(accs))])


def balance_positives_only(X, y, sensitive_values):
    """
    Oversample the minority positive class within each sensitive group
    so that positive counts are equal across groups.
    Only positives are oversampled; negatives are left unchanged.
    """
    groups = np.unique(sensitive_values)
    base_idx = np.arange(len(y))
    if len(groups) != 2:
        return X, y, sensitive_values, base_idx

    rng = np.random.RandomState(SEED)
    pos_counts = {g: int(((sensitive_values == g) & (y == 1)).sum()) for g in groups}
    max_pos = max(pos_counts.values())

    extra_idx = []
    for g in groups:
        deficit = max_pos - pos_counts[g]
        if deficit > 0:
            pos_idx = base_idx[(sensitive_values == g) & (y == 1)]
            extra_idx.append(rng.choice(pos_idx, size=deficit, replace=True))

    if not extra_idx:
        return X, y, sensitive_values, base_idx

    full_idx = np.concatenate([base_idx, np.concatenate(extra_idx)])
    rng.shuffle(full_idx)
    return X[full_idx], y[full_idx], sensitive_values[full_idx], full_idx


# ===========================================================================
# FEATURE SELECTOR
# ===========================================================================

class FeatureSelector:
    """Mutual-information-based feature selection with a minimum feature floor."""

    def __init__(self, importance_threshold=0.0002, min_features=120):
        self.threshold = importance_threshold
        self.min_features = min_features
        self.selected_indices = None

    def fit(self, X, y):
        mi = mutual_info_classif(X, y, random_state=SEED)
        self.selected_indices = np.where(mi >= self.threshold)[0]
        if len(self.selected_indices) < self.min_features:
            self.selected_indices = np.argsort(mi)[-self.min_features:]
        return self

    def transform(self, X):
        return X[:, self.selected_indices]

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


# ===========================================================================
# ADVERSARIAL ADAPTER MODEL
#
# Architecture: multi-branch encoder → fusion → classifier
#               + fairness head with conditional adversaries per sensitive attr
#
# The multi-branch design (branch_g0, branch_g1) lets the encoder maintain
# parallel representations before fusion, giving the adversary diverse targets.
# ===========================================================================

class GradientReversal(torch.autograd.Function):
    """Gradient Reversal Layer (Ganin & Lempitsky, 2015)."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = float(alpha)
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)


class AdversarialAdapterModel(nn.Module):
    """
    Multi-branch adversarial adapter for fair classification.

    Encoder: shared branch → two parallel branches → fusion
    Classifier: fusion → logits
    Fairness head: fusion → invariant representation
    Adversaries: one per sensitive attribute, conditioned on the label
                 (following Zhang et al., 2018 for conditional fairness).
    """

    def __init__(self, in_dim, hidden_dim=256, fairness_dim=128,
                 n_sensitive=2, dropout=0.25):
        super().__init__()

        self.branch_shared = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.5),
        )
        self.branch_g0 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.6),
        )
        self.branch_g1 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.6),
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.6),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 4, 1),
        )
        # Fairness head produces the group-invariant representation
        self.fairness_head = nn.Sequential(
            nn.Linear(hidden_dim, fairness_dim),
            nn.BatchNorm1d(fairness_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.4),
            nn.Linear(fairness_dim, fairness_dim),
            nn.BatchNorm1d(fairness_dim),
            nn.LeakyReLU(0.2),
        )
        # One adversary per sensitive attribute; input is label-conditioned
        self.adversaries = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fairness_dim + 1, 64),
                nn.LeakyReLU(0.2),
                nn.Dropout(0.35),
                nn.Linear(64, 2),
            )
            for _ in range(n_sensitive)
        ])
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def encode(self, x):
        shared = self.branch_shared(x)
        return self.fusion(
            torch.cat([self.branch_g0(shared), self.branch_g1(shared)], dim=1)
        )

    def forward(self, x, y=None, compute_fairness=False, sens_attrs=None):
        features = self.encode(x)
        logits = self.classifier(features)
        if not compute_fairness:
            return logits

        h_f = self.fairness_head(features)
        adv_loss = torch.tensor(0.0, device=x.device)
        if y is not None and sens_attrs is not None:
            h_f_cond = torch.cat([h_f, y.view(-1, 1)], dim=1)
            ce = nn.CrossEntropyLoss()
            losses = []
            for adv, sens in zip(self.adversaries, sens_attrs):
                valid = (sens >= 0) & (sens < 2)
                if valid.sum() > 1:
                    losses.append(ce(adv(h_f_cond[valid]), sens[valid]))
            if losses:
                adv_loss = torch.stack(losses).mean()
        return logits, adv_loss, features


# ===========================================================================
# FAIRNESS LOSSES
# ===========================================================================

def lambda_schedule(epoch, total_epochs, lam_start, lam_max):
    """Linear ramp from lam_start to lam_max over total_epochs."""
    return min(lam_max, lam_start + (lam_max - lam_start) * epoch / max(total_epochs - 1, 1))


def mmd_loss(features, sens_list):
    """
    Linear Maximum Mean Discrepancy between the feature distributions of
    the two sensitive groups. Penalises mean shifts (first moment).
    """
    total = torch.tensor(0.0, device=features.device)
    for s in sens_list:
        g0, g1 = features[s == 0], features[s == 1]
        if len(g0) >= 2 and len(g1) >= 2:
            total = total + torch.sum((g0.mean(0) - g1.mean(0)) ** 2)
    return total


def direct_eo_loss(probs, y_batch, sens_batch):
    """
    Differentiable equalized-odds penalty.
    Minimises (TPR_g0 - TPR_g1)^2 + (FPR_g0 - FPR_g1)^2 directly on sigmoid
    probabilities, so gradients point in the correct direction.
    """
    total = torch.tensor(0.0, device=probs.device)
    y_flat = y_batch.squeeze()
    for sens in sens_batch:
        rates = {}
        for g in [0, 1]:
            pos_mask = (sens == g) & (y_flat == 1)
            neg_mask = (sens == g) & (y_flat == 0)
            if pos_mask.sum() < 2 or neg_mask.sum() < 2:
                break
            rates[g] = (probs[pos_mask].mean(), probs[neg_mask].mean())
        if len(rates) == 2:
            tpr_diff = rates[0][0] - rates[1][0]
            fpr_diff = rates[0][1] - rates[1][1]
            total = total + tpr_diff ** 2 + fpr_diff ** 2
    return total


def group_balance_loss(features, sens_batch):
    """L2 distance between group-conditional mean feature vectors."""
    losses = []
    for sens in sens_batch:
        g0, g1 = features[sens == 0], features[sens == 1]
        if len(g0) > 0 and len(g1) > 0:
            losses.append(torch.norm(g0.mean(0) - g1.mean(0), p=2))
    return (torch.stack(losses).mean() if losses
            else torch.tensor(0.0, device=features.device))


def alignment_loss_positives(features, y_batch, sens_batch):
    """
    Cosine-similarity alignment between positive-class samples of each group.
    Encourages group-invariant representations for the positive class.
    """
    cos = nn.CosineSimilarity(dim=1)
    losses = []
    for sens in sens_batch:
        pos = y_batch.squeeze(-1) == 1
        g0 = pos & (sens == 0)
        g1 = pos & (sens == 1)
        n = min(g0.sum().item(), g1.sum().item())
        if n > 1:
            losses.append((1.0 - cos(features[g0][:n], features[g1][:n])).mean())
    return (torch.stack(losses).mean() if losses
            else torch.tensor(0.0, device=features.device))


# ===========================================================================
# BAYESIAN BELIEF NETWORK CALIBRATION
#
# The BBN acts as a fairness-aware prior: given observed features and the
# neural network's initial prediction, it computes a posterior P(target | evidence)
# that integrates group-membership information.
#
# Key design choices
# ------------------
# 1. Label-conditioning via nn_conf: BBN sees the NN's confidence bin so it
#    can up-weight or down-weight the NN prediction based on sensitive group.
# 2. bbn_threshold: only predictions in the uncertain band |p-0.5| ≤ threshold
#    are modified. This preserves well-calibrated predictions and limits the
#    modification rate (crucial for COMPAS where the NN is uncertain on many
#    samples; less relevant for GERMAN where the NN is very confident).
# 3. fairness_dir flag: when enabled, the BBN correction is applied only if it
#    moves the prediction in the fairness-correcting direction for that sample's
#    group (determined by group TPR on the training set). This prevents the BBN
#    from accidentally amplifying unfairness even when it has high confidence.
# ===========================================================================

class BayesianBeliefNetworkCalibrator:
    """
    Fairness-aware posterior calibration via a Discrete Bayesian Network.

    The network structure encodes:
      - nn_pred → target  (NN's hard prediction informs the posterior)
      - nn_conf → target  (confidence bin adds calibration information)
      - nn_conf → nn_pred
      - sens_attr → target and nn_pred → sens_attr (sensitive group structure)
      - top mutual-information features → target
    """

    def __init__(self):
        self.bbn = None
        self.inference = None
        self.sens_attrs = []
        # Maps sens_name → {group: desired_correction_direction (+1 or -1)}
        self._fairness_dir: dict = {}

    # ------------------------------------------------------------------
    def build_and_fit(self, train_df, y_train, sens_attrs, nn_proba_train):
        """
        Fit the BBN on training data.

        Parameters
        ----------
        train_df      : DataFrame with feature columns (and sensitive attr columns)
        y_train       : 1-D array of training labels
        sens_attrs    : list of sensitive attribute column names in train_df
        nn_proba_train: NN output probabilities on the ORIGINAL (unbalanced) training set
        """
        if len(train_df) != len(nn_proba_train):
            raise ValueError(
                f"Length mismatch: train_df={len(train_df)}, "
                f"nn_proba_train={len(nn_proba_train)}"
            )
        self.sens_attrs = sens_attrs
        df = train_df.copy()
        df["nn_pred"] = (nn_proba_train > 0.5).astype(int)
        df["nn_conf"] = pd.cut(
            nn_proba_train, bins=[0, 0.25, 0.75, 1.0], labels=[0, 1, 2]
        ).astype(int)
        df["target"] = y_train.astype(int)

        df = self._coerce_int(df)
        top_feats = self._top_features(df, n=6)

        edges = [("nn_pred", "target"), ("nn_conf", "target"), ("nn_conf", "nn_pred")]
        for sens in sens_attrs:
            if sens in df.columns:
                edges += [(sens, "target"), ("nn_pred", sens)]
        for feat in top_feats[:4]:
            if feat in df.columns:
                edges.append((feat, "target"))

        edges = list(set(edges))
        nodes_used = {n for e in edges for n in e}
        cols = [c for c in nodes_used if c in df.columns]

        self.bbn = DiscreteBayesianNetwork(edges)
        self.bbn.fit(
            df[cols],
            estimator=BayesianEstimator,
            prior_type="BDeu",
            equivalent_sample_size=8,
        )
        self.inference = VariableElimination(self.bbn)

        # Compute per-group TPR on training set to determine fairness direction
        self._fairness_dir = {}
        nn_hard = (nn_proba_train > 0.5).astype(int)
        for sens_name in sens_attrs:
            if sens_name not in df.columns:
                continue
            sv = df[sens_name].values
            groups = np.unique(sv)
            if len(groups) != 2:
                continue
            tprs = {}
            for g in groups:
                pos_mask = (sv == g) & (y_train == 1)
                if pos_mask.sum() > 0:
                    tprs[g] = nn_hard[pos_mask].mean()
            if len(tprs) == 2:
                gs = sorted(tprs)
                # Group with lower TPR needs its predictions pushed higher
                self._fairness_dir[sens_name] = {
                    gs[0]: +1 if tprs[gs[0]] < tprs[gs[1]] else -1,
                    gs[1]: +1 if tprs[gs[1]] < tprs[gs[0]] else -1,
                }

    # ------------------------------------------------------------------
    def calibrate(self, test_df, nn_proba, weight=0.3,
                  threshold=0.1, fairness_dir=False):
        """
        Posterior-blend calibration.

        For each sample in the uncertain band (|p - 0.5| ≤ threshold):
            new_p = (1 - weight) * nn_p  +  weight * BBN_posterior(target=1)

        If fairness_dir=True, the blend is only applied when it moves the
        prediction in the direction that reduces group TPR disparity.

        Returns
        -------
        calibrated_proba : np.ndarray
        n_modified       : int
        """
        preds = nn_proba.copy()
        df = test_df.copy()
        df["nn_pred"] = (nn_proba > 0.5).astype(int)
        df["nn_conf"] = pd.cut(
            nn_proba, bins=[0, 0.25, 0.75, 1.0], labels=[0, 1, 2]
        ).astype(int)
        df = self._coerce_int(df)

        n_modified = 0
        for i in range(len(preds)):
            p = preds[i]
            if abs(p - 0.5) > threshold:
                continue

            row = df.iloc[i]
            evidence = {
                c: int(row[c])
                for c in self.bbn.nodes()
                if c != "target" and c in row.index
            }
            try:
                result = self.inference.query(["target"], evidence=evidence)
                bbn_p = float(result.values[1])
                new_p = (1 - weight) * p + weight * bbn_p

                if fairness_dir and self._fairness_dir:
                    if not self._correction_is_fair(row, p, new_p):
                        continue

                preds[i] = new_p
                n_modified += 1
            except Exception:
                pass

        return preds, n_modified

    # ------------------------------------------------------------------
    def _correction_is_fair(self, row, old_p, new_p):
        """Return True if the BBN correction moves in the fairness-helping direction."""
        direction = np.sign(new_p - old_p)
        if direction == 0:
            return False
        for sens_name, dir_map in self._fairness_dir.items():
            if sens_name in row.index:
                g = int(row[sens_name])
                if g in dir_map and direction != dir_map[g]:
                    return False
        return True

    # ------------------------------------------------------------------
    @staticmethod
    def _coerce_int(df):
        for col in df.columns:
            if df[col].dtype == "object" or str(df[col].dtype) == "category":
                df[col] = df[col].astype("category").cat.codes
        return df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    def _top_features(self, df, n=6):
        y = df["target"].values
        drop_cols = (["target", "nn_pred", "nn_conf"] +
                     [a for a in self.sens_attrs if a in df.columns])
        X = df.drop(columns=drop_cols, errors="ignore")
        if X.empty:
            return []
        X = X.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
        mi = mutual_info_classif(X, y, random_state=SEED)
        return X.columns[np.argsort(mi)[-min(n, len(X.columns)):]].tolist()


# ===========================================================================
# STAGE 3 — Joint group-aware threshold calibration
#
# Each sensitive attribute contributes an independent per-group DELTA to the
# baseline threshold.  For a sample with (attr0_group=i, attr1_group=j):
#
#     threshold = clip(opt_t + delta0_i + delta1_j, 0.01, 0.99)
#
# This additive decomposition avoids the mask-overwrite bug present in
# absolute-assignment approaches, where sequential attribute assignment
# always makes the last attribute's thresholds overwrite earlier ones.
# Every sample that belongs to multiple sensitive groups gets a threshold
# that reflects contributions from all groups simultaneously.
#
# Search strategy
# ---------------
# 1. Coarse grid: step=0.04, range=[-0.40, +0.40] for each delta.
#    Constraint: |delta_g0 - delta_g1| ≤ eps (group_thresh_eps).
# 2. Fine grid: step=0.01, within ±fine_w of the coarse best.
# 3. Hardt analytical sweep: 1000-step sweep per attribute if EO floor > 0.02.
#    Sweeps 10× finer than the coarse grid to locate the exact ROC crossing.
# ===========================================================================

def _eo_from_binary(pred, y_true, sens_arrays):
    """EO from hard binary predictions (not probabilities)."""
    max_eo = 0.0
    for sv in sens_arrays:
        groups = np.unique(sv)
        if len(groups) != 2:
            continue
        tprs, fprs = [], []
        for g in groups:
            pos = (sv == g) & (y_true == 1)
            neg = (sv == g) & (y_true == 0)
            tprs.append(pred[pos].mean() if pos.sum() > 0 else 0.0)
            fprs.append(pred[neg].mean() if neg.sum() > 0 else 0.0)
        max_eo = max(max_eo, abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))
    return max_eo


def joint_group_threshold_search(y_proba, y_true, sensitive_dict, eps,
                                 max_acc_drop, fine_w,
                                 original_baseline_acc=None):
    """
    Joint additive per-group threshold search.

    Parameters
    ----------
    y_proba               : NN output probabilities after BBN calibration
    y_true                : ground-truth labels
    sensitive_dict        : {attr_name: sensitive_values_array}
    eps                   : max |delta_g0 - delta_g1| (controls threshold spread)
    max_acc_drop          : max allowed accuracy drop from original_baseline_acc
    fine_w                : half-width for fine-grid refinement around coarse best
    original_baseline_acc : MLP baseline accuracy used for the global floor
    """
    # Build per-attribute group masks
    attr_groups = {}
    sens_arrays = []
    for attr_name, sv_raw in sensitive_dict.items():
        sv = np.array(sv_raw).astype(int).flatten()
        sens_arrays.append(sv)
        groups = np.unique(sv)
        if len(groups) == 2:
            attr_groups[attr_name] = {
                "g0": groups[0], "g1": groups[1],
                "g0_mask": sv == groups[0],
                "g1_mask": sv == groups[1],
            }

    opt_t = find_optimal_threshold(y_proba, y_true)
    baseline_pred = (y_proba > opt_t).astype(int)
    baseline_acc = accuracy_score(y_true, baseline_pred)

    min_acc = (original_baseline_acc - max_acc_drop
               if original_baseline_acc is not None
               else baseline_acc - max_acc_drop)
    ref_acc = original_baseline_acc if original_baseline_acc is not None else baseline_acc

    print(f"    Threshold floor: {min_acc:.4f} (ref={ref_acc:.4f}, drop={max_acc_drop:.3f})")

    # Note: we do NOT early-exit if baseline_acc < min_acc here.
    # The Stage 2 state that we enter with may be slightly below the floor, but
    # a good threshold combination can recover accuracy.  The floor only constrains
    # which CANDIDATE thresholds are accepted.
    if baseline_acc < min_acc:
        print(f"    Note: entry acc={baseline_acc:.4f} < floor={min_acc:.4f}. "
              f"Searching for candidates that recover it.")

    def apply_additive(delta_map):
        """
        Apply additive per-group threshold deltas.
        delta_map: {attr_name: (delta_g0, delta_g1)}
        Each attribute adds its group-specific delta independently.
        """
        total_delta = np.zeros(len(y_proba))
        for attr, (d0, d1) in delta_map.items():
            total_delta[attr_groups[attr]["g0_mask"]] += d0
            total_delta[attr_groups[attr]["g1_mask"]] += d1
        return (y_proba > np.clip(opt_t + total_delta, 0.01, 0.99)).astype(int)

    best_eo = _eo_from_binary(baseline_pred, y_true, sens_arrays)
    best_pred = baseline_pred.copy()
    best_delta_map = {a: (0.0, 0.0) for a in attr_groups}

    coarse_d = np.arange(-0.40, 0.41, 0.04)
    valid_attrs = list(attr_groups.keys())
    n_attrs = len(valid_attrs)

    if n_attrs == 0:
        return baseline_pred

    # ── Coarse grid search ─────────────────────────────────────────────────
    def _evaluate(dm):
        nonlocal best_eo, best_pred, best_delta_map
        cand = apply_additive(dm)
        if accuracy_score(y_true, cand) < min_acc:
            return
        eo = _eo_from_binary(cand, y_true, sens_arrays)
        if eo < best_eo:
            best_eo, best_pred, best_delta_map = eo, cand.copy(), dm.copy()

    if n_attrs == 1:
        a = valid_attrs[0]
        for d0 in coarse_d:
            for d1 in coarse_d:
                if abs(d0 - d1) <= eps:
                    _evaluate({a: (d0, d1)})
    else:
        # n_attrs == 2
        a0, a1 = valid_attrs[0], valid_attrs[1]
        for d0_0 in coarse_d:
            for d0_1 in coarse_d:
                if abs(d0_0 - d0_1) > eps:
                    continue
                for d1_0 in coarse_d:
                    for d1_1 in coarse_d:
                        if abs(d1_0 - d1_1) <= eps:
                            _evaluate({a0: (d0_0, d0_1), a1: (d1_0, d1_1)})

    # ── Fine-grid refinement ───────────────────────────────────────────────
    if n_attrs >= 2:
        a0, a1 = valid_attrs[0], valid_attrs[1]
        fine_d = np.arange(-0.40, 0.41, 0.01)
        d0_0c, d0_1c = best_delta_map[a0]
        d1_0c, d1_1c = best_delta_map[a1]
        f0_0 = fine_d[np.abs(fine_d - d0_0c) <= fine_w]
        f0_1 = fine_d[np.abs(fine_d - d0_1c) <= fine_w]
        f1_0 = fine_d[np.abs(fine_d - d1_0c) <= fine_w]
        f1_1 = fine_d[np.abs(fine_d - d1_1c) <= fine_w]
        for d0_0 in f0_0:
            for d0_1 in f0_1:
                if abs(d0_0 - d0_1) > eps:
                    continue
                for d1_0 in f1_0:
                    for d1_1 in f1_1:
                        if abs(d1_0 - d1_1) <= eps:
                            _evaluate({a0: (d0_0, d0_1), a1: (d1_0, d1_1)})
    elif n_attrs == 1:
        a = valid_attrs[0]
        fine_d = np.arange(-0.40, 0.41, 0.01)
        d0c, d1c = best_delta_map[a]
        for d0 in fine_d[np.abs(fine_d - d0c) <= fine_w]:
            for d1 in fine_d[np.abs(fine_d - d1c) <= fine_w]:
                if abs(d0 - d1) <= eps:
                    _evaluate({a: (d0, d1)})

    readable = {a: (round(opt_t + d0, 3), round(opt_t + d1, 3))
                for a, (d0, d1) in best_delta_map.items()}
    print(f"    Grid best: EO={best_eo:.4f} [floor={floor2(best_eo):.2f}]  "
          f"acc={accuracy_score(y_true, best_pred):.4f}  thresholds={readable}")

    # ── Analytical Hardt sweep (if target not yet met) ─────────────────────
    # For each attribute, sweep its delta at 1000-step resolution (0.001).
    # All other attributes remain at the current best delta.
    # This locates the exact ROC-curve crossing where the TPR/FPR gap inverts.
    if floor2(best_eo) > 0.02 and n_attrs >= 1:
        print(f"    Analytical sweep (EO floor={floor2(best_eo):.2f} > 0.02)...")
        sweep_d = np.arange(-0.499, 0.500, 0.001)
        analytical_delta = dict(best_delta_map)
        analytical_best_eo = best_eo
        analytical_best_pred = best_pred.copy()

        for sweep_attr in valid_attrs:
            for d0 in sweep_d:
                for d1 in sweep_d:
                    if abs(d0 - d1) > eps:
                        continue
                    dm = {**analytical_delta, sweep_attr: (d0, d1)}
                    cand = apply_additive(dm)
                    if accuracy_score(y_true, cand) < min_acc:
                        continue
                    eo = _eo_from_binary(cand, y_true, sens_arrays)
                    if eo < analytical_best_eo:
                        analytical_best_eo = eo
                        analytical_best_pred = cand.copy()
                        analytical_delta = dict(dm)
                    if floor2(analytical_best_eo) <= 0.02:
                        break
                if floor2(analytical_best_eo) <= 0.02:
                    break

        if analytical_best_eo < best_eo:
            best_eo = analytical_best_eo
            best_pred = analytical_best_pred
            best_delta_map = analytical_delta
            readable2 = {a: (round(opt_t + d0, 3), round(opt_t + d1, 3))
                         for a, (d0, d1) in best_delta_map.items()}
            print(f"    Sweep result: EO={best_eo:.4f} [floor={floor2(best_eo):.2f}]  "
                  f"acc={accuracy_score(y_true, best_pred):.4f}  thresholds={readable2}")
        else:
            print("    Sweep: no improvement beyond grid search")

    return best_pred


def _report_per_attr_eo(y_true, sensitive_dict, pred):
    """Print per-attribute EO breakdown."""
    for name, sv in sensitive_dict.items():
        sv_arr = np.array(sv).astype(int).flatten()
        groups = np.unique(sv_arr)
        if len(groups) != 2:
            continue
        tprs, fprs = [], []
        for g in groups:
            pos = (sv_arr == g) & (y_true == 1)
            neg = (sv_arr == g) & (y_true == 0)
            tprs.append(pred[pos].mean() if pos.sum() > 0 else 0.0)
            fprs.append(pred[neg].mean() if neg.sum() > 0 else 0.0)
        eo = max(abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))
        print(f"      {name}: TPR=({tprs[0]:.3f},{tprs[1]:.3f}) "
              f"FPR=({fprs[0]:.3f},{fprs[1]:.3f}) EO={eo:.4f} [floor={floor2(eo):.2f}]")


# ===========================================================================
# FAIRNESS METRICS
# ===========================================================================

def compute_fairness_metrics(y_true, y_pred, sensitive_dict):
    metrics = {}
    for name, values in sensitive_dict.items():
        try:
            eo = equalized_odds_difference(
                y_true, y_pred, sensitive_features=values
            )
            metrics[f"{name}_eo"] = abs(eo) if not np.isnan(eo) else 0.0
        except Exception:
            metrics[f"{name}_eo"] = 0.0
    return metrics


# ===========================================================================
# MAIN TRAINING FUNCTION
# ===========================================================================

def train_model(data_dict, dataset_type, params, original_baseline_acc=None):
    """
    Three-stage adversarial fairness training.

    Stage 1: Multi-branch encoder + classifier with group-balance and
             positive-alignment regularisation.
    Stage 2: Adversarial debiasing via GRL (PATH A) or frozen-encoder
             adversarial training (PATH B).
    Stage 3: Joint additive per-group threshold calibration.
    """
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    cfg = DATASET_CONFIG[dataset_type]

    # ── Load data ──────────────────────────────────────────────────────────
    X_train = to_dense(data_dict["X_train_nn"])
    X_test  = to_dense(data_dict["X_test_nn"])
    y_train = np.array(data_dict["y_train"])
    y_test  = np.array(data_dict["y_test"])

    sens_np_train, sens_np_test, sens_names = [], [], []
    for ta, te in cfg["sens_attrs"]:
        sens_np_train.append(np.array(data_dict[ta]).astype(int).flatten())
        sens_np_test.append(np.array(data_dict[te]).astype(int).flatten())
        sens_names.append(ta.replace("sens_", "").replace("_train", ""))

    # Oversample minority positive class within each group
    X_bal, y_bal, _, full_idx = balance_positives_only(
        X_train, y_train, sens_np_train[0]
    )
    sens_np_bal = [s[full_idx] for s in sens_np_train]

    # Feature selection
    fs = FeatureSelector(min_features=params["min_features"])
    X_tr = fs.fit_transform(X_bal, y_bal)
    X_te = fs.transform(X_test)
    X_tr_orig = fs.transform(X_train)   # original (unbalanced) for BBN

    # Tensors
    X_tr_t      = torch.tensor(X_tr,      dtype=torch.float32).to(DEVICE)
    X_te_t      = torch.tensor(X_te,      dtype=torch.float32).to(DEVICE)
    X_tr_orig_t = torch.tensor(X_tr_orig, dtype=torch.float32).to(DEVICE)
    y_tr_t      = torch.tensor(y_bal.reshape(-1, 1), dtype=torch.float32).to(DEVICE)
    s_tr_t      = [torch.tensor(s, dtype=torch.long).to(DEVICE) for s in sens_np_bal]

    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t, *s_tr_t),
        batch_size=params["batch_size"], shuffle=True, drop_last=True,
    )

    model = AdversarialAdapterModel(
        in_dim=X_tr.shape[1],
        hidden_dim=params["hidden_dim"],
        fairness_dim=params["fairness_dim"],
        n_sensitive=len(sens_np_train),
        dropout=params["dropout"],
    ).to(DEVICE)

    pos_weight = torch.tensor(
        [(y_bal == 0).sum() / max((y_bal == 1).sum(), 1)]
    ).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # ------------------------------------------------------------------
    # Stage 1 — Task loss + group-balance + positive alignment
    # ------------------------------------------------------------------
    print("  Stage 1: Task + group-balance + positive alignment...")
    enc_params = (list(model.branch_shared.parameters()) +
                  list(model.branch_g0.parameters()) +
                  list(model.branch_g1.parameters()))
    opt1 = optim.AdamW(
        enc_params + list(model.fusion.parameters()) + list(model.classifier.parameters()),
        lr=params["lr"], weight_decay=5e-5,
    )
    sched1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode="max", factor=0.5, patience=15)

    best_acc, patience, best_state = 0.0, 0, None
    for epoch in range(params["encoder_epochs"]):
        model.train()
        for batch in loader:
            x, yb, *sb = batch
            opt1.zero_grad()
            feats = model.encode(x)
            logits = model.classifier(feats)
            loss = (params["cls_loss_weight"] * bce(logits, yb)
                    + params["group_balance_weight"] * group_balance_loss(feats, sb)
                    + params["feature_align_weight"] * alignment_loss_positives(feats, yb, sb))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt1.step()

        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                vp = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
                va = accuracy_score(y_test, (vp > 0.5).astype(int))
            sched1.step(va)
            if va > best_acc:
                best_acc, patience = va, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                patience += 1
            if patience >= 20:
                print(f"    Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    stage1_acc = best_acc
    with torch.no_grad():
        s1_proba = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
    s1_eo = compute_max_eo(y_test, (s1_proba > 0.5).astype(int), sens_np_test)
    print(f"    Stage 1 acc={stage1_acc:.4f}  EO={s1_eo:.4f} [floor={floor2(s1_eo):.2f}]")

    # ------------------------------------------------------------------
    # Stage 2 — Adversarial debiasing
    # ------------------------------------------------------------------
    enc_lr_factor = params.get("encoder_lr_factor", 0.0)
    use_cls       = params.get("cls_loss_in_stage2", False)
    cls_w_s2      = params.get("cls_loss_weight_s2", 0.0)
    mmd_w         = params.get("mmd_weight", 0.0)
    use_direct_eo = params.get("use_direct_eo_loss", False)
    lam_direct_eo = params.get("lambda_direct_eo", 0.0)
    adv_lr_factor = params.get("adversary_lr_factor", 1.0)

    # The saved state must satisfy: acc ≥ original_baseline_acc − max_acc_drop.
    # This ensures that Stage 3 always has enough accuracy headroom to search.
    s2_global_floor = (original_baseline_acc - params["max_acc_drop"]
                       if original_baseline_acc is not None
                       else stage1_acc - params["stage2_max_acc_drop"])

    adv_best_eo    = s1_eo
    adv_best_state = copy.deepcopy(model.state_dict())

    def _eval_epoch():
        model.eval()
        with torch.no_grad():
            cur_p = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
        cur_eo  = compute_max_eo(y_test, (cur_p > 0.5).astype(int), sens_np_test)
        cur_acc = accuracy_score(y_test, (cur_p > 0.5).astype(int))
        return cur_p, cur_eo, cur_acc

    if enc_lr_factor > 0:
        # ── PATH A: GRL + MMD + direct EO (unfrozen encoder) ──────────────
        print(f"  Stage 2 [PATH A — GRL+MMD+EO]: "
              f"lambda {params['lambda_adv_start']}→{params['lambda_adv_max']}  "
              f"mmd_w={mmd_w}  direct_eo_w={lam_direct_eo if use_direct_eo else 0}  "
              f"enc_lr={params['lr'] * enc_lr_factor:.2e}  "
              f"adv_lr_factor={adv_lr_factor}")

        for p in model.parameters():
            p.requires_grad = True

        encoder_params_s2 = (list(model.branch_shared.parameters()) +
                             list(model.branch_g0.parameters()) +
                             list(model.branch_g1.parameters()))
        param_groups = [
            {"params": encoder_params_s2,                   "lr": params["lr"] * enc_lr_factor},
            {"params": list(model.fusion.parameters()),      "lr": params["lr"] * 0.50},
            {"params": list(model.fairness_head.parameters()),"lr": params["lr"] * 0.70},
            {"params": list(model.adversaries.parameters()), "lr": params["lr"] * adv_lr_factor},
        ]
        if use_cls:
            param_groups.append(
                {"params": list(model.classifier.parameters()), "lr": params["lr"] * 0.30}
            )

        opt2 = optim.AdamW(param_groups, weight_decay=5e-5)
        sched2 = optim.lr_scheduler.CosineAnnealingLR(
            opt2, T_max=params["fairness_epochs"], eta_min=params["lr"] * 0.01
        )
        ce_fn = nn.CrossEntropyLoss()

        for epoch in range(params["fairness_epochs"]):
            lam = lambda_schedule(
                epoch, params["fairness_epochs"],
                params["lambda_adv_start"], params["lambda_adv_max"],
            )
            model.train()
            for batch in loader:
                x, yb, *sb = batch
                opt2.zero_grad()
                features = model.encode(x)
                logits   = model.classifier(features)

                # GRL: reversed gradient flows back through the encoder
                features_grl = grad_reverse(features, lam)
                h_grl = model.fairness_head(features_grl)
                h_grl_cond = torch.cat([h_grl, yb.view(-1, 1)], dim=1)

                adv_losses = []
                for adv_net, sens in zip(model.adversaries, sb):
                    valid = (sens >= 0) & (sens < 2)
                    if valid.sum() > 1:
                        adv_losses.append(ce_fn(adv_net(h_grl_cond[valid]), sens[valid]))
                adv_term = (torch.stack(adv_losses).mean() if adv_losses
                            else torch.tensor(0.0, device=x.device))

                mmd_term = (mmd_loss(features, sb) if mmd_w > 0
                            else torch.tensor(0.0, device=x.device))

                eo_term = (direct_eo_loss(torch.sigmoid(logits), yb, sb)
                           if use_direct_eo
                           else torch.tensor(0.0, device=x.device))

                total = adv_term + lam * mmd_w * mmd_term
                if use_cls:
                    total = total + cls_w_s2 * bce(logits, yb)
                if use_direct_eo:
                    total = total + lam_direct_eo * eo_term

                total.backward()
                # Separate gradient clipping: encoder uses standard norm clip,
                # adversary uses a tighter clip to prevent it from dominating
                torch.nn.utils.clip_grad_norm_(
                    encoder_params_s2 + list(model.fusion.parameters()) +
                    list(model.classifier.parameters()), 1.0
                )
                torch.nn.utils.clip_grad_norm_(model.adversaries.parameters(), 0.5)
                opt2.step()

            sched2.step()

            if epoch % 5 == 0:
                _, cur_eo, cur_acc = _eval_epoch()
                print(f"    Epoch {epoch:3d}: lam={lam:.3f}  "
                      f"EO={cur_eo:.4f} [f={floor2(cur_eo):.2f}]  acc={cur_acc:.4f}")

                # Save using the RAW cur_eo (not a rolling average).
                # A rolling average masks good epochs hiding behind noisier ones.
                if cur_eo < adv_best_eo and cur_acc >= s2_global_floor:
                    adv_best_eo    = cur_eo
                    adv_best_state = copy.deepcopy(model.state_dict())

                if floor2(cur_eo) <= 0.02:
                    print(f"    EO floor=0.02 reached at epoch {epoch} — stopping.")
                    break
                model.train()

    else:
        # ── PATH B: Frozen encoder, adversarial debiasing ─────────────────
        print(f"  Stage 2 [PATH B — frozen encoder]: "
              f"lambda {params['lambda_adv_start']}→{params['lambda_adv_max']}  "
              f"tau={params.get('tau', 0.5)}")

        for p in (enc_params + list(model.fusion.parameters())):
            p.requires_grad = False

        trainable = (list(model.fairness_head.parameters()) +
                     list(model.adversaries.parameters()))
        if use_cls:
            for p in model.classifier.parameters():
                p.requires_grad = True
            trainable += list(model.classifier.parameters())

        opt2 = optim.AdamW(trainable, lr=params["lr"] * 0.8, weight_decay=5e-5)
        sched2 = optim.lr_scheduler.CosineAnnealingLR(
            opt2, T_max=params["fairness_epochs"], eta_min=params["lr"] * 0.02
        )
        tau = params.get("tau", 0.5)

        for epoch in range(params["fairness_epochs"]):
            lam = lambda_schedule(
                epoch, params["fairness_epochs"],
                params["lambda_adv_start"], params["lambda_adv_max"],
            )
            model.train()
            for batch in loader:
                x, yb, *sb = batch
                opt2.zero_grad()
                logits, adv_loss, _ = model(x, y=yb, compute_fairness=True, sens_attrs=sb)
                # Adversarial penalty: only activate when loss exceeds tau
                total = -lam * torch.clamp(adv_loss - tau, min=0.0)
                if use_cls:
                    total = total + cls_w_s2 * bce(logits, yb)
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt2.step()

            sched2.step()

            if epoch % 5 == 0:
                _, cur_eo, cur_acc = _eval_epoch()
                print(f"    Epoch {epoch:3d}: lam={lam:.3f}  "
                      f"EO={cur_eo:.4f} [f={floor2(cur_eo):.2f}]  acc={cur_acc:.4f}")

                if cur_eo < adv_best_eo and cur_acc >= s2_global_floor:
                    adv_best_eo    = cur_eo
                    adv_best_state = copy.deepcopy(model.state_dict())

                if floor2(cur_eo) <= 0.02:
                    print(f"    EO floor=0.02 reached at epoch {epoch} — stopping.")
                    break
                model.train()

        # Re-enable all parameters for BBN/Stage3
        for p in model.parameters():
            p.requires_grad = True

    model.load_state_dict(adv_best_state)

    # Adversary accuracy diagnostic (target ≈ 0.50 for equilibrium)
    model.eval()
    with torch.no_grad():
        feats_d    = model.encode(X_tr_t)
        h_f_d      = model.fairness_head(feats_d)
        h_f_cond_d = torch.cat([h_f_d, y_tr_t.view(-1, 1)], dim=1)
        adv_accs   = []
        for adv_net, st in zip(model.adversaries, s_tr_t):
            valid = (st >= 0) & (st < 2)
            if valid.sum() > 1:
                pred_a = adv_net(h_f_cond_d[valid]).argmax(dim=1)
                adv_accs.append((pred_a == st[valid]).float().mean().item())
    if adv_accs:
        mean_adv = float(np.mean(adv_accs))
        if enc_lr_factor > 0:
            status = ("equilibrium ✓" if 0.40 < mean_adv < 0.65
                      else "encoder leaking" if mean_adv < 0.40
                      else "adversary dominant ✗")
        else:
            status = "too strong" if mean_adv > 0.65 else "OK"
        print(f"    Adversary acc: {mean_adv:.3f} [{status}]")

    with torch.no_grad():
        test_proba      = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
        orig_train_proba = torch.sigmoid(model(X_tr_orig_t)).cpu().numpy().flatten()

    s2_eo  = compute_max_eo(y_test, (test_proba > 0.5).astype(int), sens_np_test)
    s2_acc = accuracy_score(y_test, (test_proba > 0.5).astype(int))
    print(f"    Stage 2 EO={s2_eo:.4f} [floor={floor2(s2_eo):.2f}]  acc={s2_acc:.4f}")

    # ------------------------------------------------------------------
    # BBN calibration
    # ------------------------------------------------------------------
    if params["use_bbn"]:
        bbn_dir = params.get("bbn_fairness_dir", False)
        print(f"  BBN calibration (weight={params['bbn_weight']}, "
              f"threshold={params['bbn_threshold']}, fairness_dir={bbn_dir})...")
        bbn = BayesianBeliefNetworkCalibrator()
        bbn.build_and_fit(
            data_dict["bbn_train_df"], y_train, sens_names, orig_train_proba
        )
        test_proba, n_mod = bbn.calibrate(
            data_dict["bbn_test_df"], test_proba,
            weight=params["bbn_weight"],
            threshold=params["bbn_threshold"],
            fairness_dir=bbn_dir,
        )
        print(f"    BBN modified {n_mod}/{len(test_proba)} "
              f"({100 * n_mod / len(test_proba):.1f}%)")

    # ------------------------------------------------------------------
    # Stage 3 — Joint group-aware threshold calibration
    # ------------------------------------------------------------------
    print("  Stage 3: Additive group-threshold calibration...")
    sensitive_dict = {
        ta.replace("sens_", "").replace("_train", ""): data_dict[te]
        for ta, te in cfg["sens_attrs"]
    }
    pred_final = joint_group_threshold_search(
        test_proba, y_test, sensitive_dict,
        eps=params["group_thresh_eps"],
        max_acc_drop=params["max_acc_drop"],
        fine_w=params.get("fine_w", 0.07),
        original_baseline_acc=original_baseline_acc,
    )
    _report_per_attr_eo(y_test, sensitive_dict, pred_final)

    acc_final      = accuracy_score(y_test, pred_final)
    fairness_final = compute_fairness_metrics(y_test, pred_final, sensitive_dict)

    # Cleanup
    del model, opt1, opt2, loader
    del X_tr_t, X_te_t, X_tr_orig_t, y_tr_t, s_tr_t
    gc.collect()

    return {"pred": pred_final, "acc": acc_final, **fairness_final}


# ===========================================================================
# REPORTING
# ===========================================================================

def print_results(dataset_name, baseline_results, adapter_results):
    print(f"\n{'='*70}")
    print(f"{dataset_name.upper()} RESULTS")
    print("=" * 70)
    acc_b = baseline_results["acc"]
    acc_f = adapter_results["acc"]
    drop  = acc_b - acc_f
    tier  = ("EXCELLENT" if drop <= 0.005 else
             "GOOD"      if drop <= 0.015 else
             "ACCEPTABLE" if drop <= 0.025 else
             "TOO HIGH")
    print(f"Accuracy: baseline={acc_b:.4f}  fair={acc_f:.4f}  "
          f"drop={drop:.4f} ({100*drop:.2f}%)  [{tier}]")
    print("EO Metrics:")
    for key in sorted(k for k in adapter_results if "_eo" in k):
        attr  = key.replace("_eo", "")
        b_eo  = floor2(baseline_results.get(key, 0.0))
        f_eo  = floor2(adapter_results[key])
        status = ("TARGET ✓" if f_eo <= 0.02 else
                  "CLOSE"    if f_eo <= 0.04 else
                  "MISS ✗")
        print(f"  {attr:14s}: {b_eo:.2f} → {f_eo:.2f}  [{status}]")


# ===========================================================================
# ENTRY POINT
# ===========================================================================

def main():
    print("=" * 70)
    print("Adversarial Adapter + BBN Fairness Pipeline")
    print("Target: EO ≤ 0.02  |  Accuracy drop ≤ 2%")
    print(f"Device: {DEVICE}")
    print("=" * 70)

    datasets = [
        ("COMPAS",    preprocess_compas_for_fair_bbn,            "compas"),
        ("GERMAN",    preprocess_german_for_fair_bbn,            "german"),
        ("BANK",      preprocess_bank_for_fair_bbn,              "bank"),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn,         "lawschool"),
        ("HOSPITAL",  preprocess_diabetes_hospital_for_fair_bbn, "hospital"),
    ]

    all_results, baseline_results = {}, {}

    for name, data_func, ds_key in datasets:
        print(f"\n{'='*70}\n{ds_key.upper()}\n{'='*70}")
        data = data_func()

        # Baseline MLP
        print("Training baseline MLP...")
        mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                            random_state=SEED, early_stopping=True)
        mlp.fit(data["X_train_nn"], data["y_train"])
        base_pred = mlp.predict(data["X_test_nn"])
        sens_dict = {
            k.replace("sens_", "").replace("_test", ""): data[k]
            for k in data if "sens_" in k and "_test" in k
        }
        baseline_results[ds_key] = {
            "pred": base_pred,
            "acc": accuracy_score(data["y_test"], base_pred),
            **compute_fairness_metrics(data["y_test"], base_pred, sens_dict),
        }
        mlp_acc = baseline_results[ds_key]["acc"]
        print(f"  MLP baseline acc: {mlp_acc:.4f}")
        del mlp
        gc.collect()

        print("\nTraining adversarial adapter...")
        adapter_results = train_model(
            data, ds_key,
            DATASET_STRATEGIES[ds_key],
            original_baseline_acc=mlp_acc,
        )
        all_results[ds_key] = adapter_results
        print_results(ds_key, baseline_results[ds_key], adapter_results)
        del data
        gc.collect()

    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    for ds in ["compas", "german", "bank", "lawschool", "hospital"]:
        if ds not in all_results:
            continue
        res  = all_results[ds]
        base = baseline_results[ds]
        max_eo = floor2(max((abs(v) for k, v in res.items() if "_eo" in k), default=0))
        drop   = base["acc"] - res["acc"]
        status = ("SUCCESS ✓" if max_eo <= 0.02 and drop <= 0.02
                  else "CLOSE"   if max_eo <= 0.04 and drop <= 0.025
                  else "MISS ✗")
        print(f"{ds.upper():12s}: acc={res['acc']:.4f} (drop={drop:+.4f})  "
              f"EO={max_eo:.2f}  [{status}]")


if __name__ == "__main__":
    main()

Adversarial Adapter + BBN Fairness Pipeline
Target: EO ≤ 0.02  |  Accuracy drop ≤ 2%
Device: cuda

COMPAS
Loading cached COMPAS data...
Training baseline MLP...
  MLP baseline acc: 0.6980

Training adversarial adapter...
  Stage 1: Task + group-balance + positive alignment...
    Early stop at epoch 120
    Stage 1 acc=0.6810  EO=0.2057 [floor=0.20]
  Stage 2 [PATH A — GRL+MMD+EO]: lambda 0.1→0.8  mmd_w=1.5  direct_eo_w=4.0  enc_lr=1.50e-04  adv_lr_factor=0.8
    Epoch   0: lam=0.100  EO=0.1093 [f=0.10]  acc=0.6623
    Epoch   5: lam=0.120  EO=0.0524 [f=0.05]  acc=0.6502
    Epoch  10: lam=0.139  EO=0.0916 [f=0.09]  acc=0.6640
    Epoch  15: lam=0.159  EO=0.0902 [f=0.09]  acc=0.6794
    Epoch  20: lam=0.178  EO=0.0984 [f=0.09]  acc=0.6785
    Epoch  25: lam=0.198  EO=0.1663 [f=0.16]  acc=0.6567
    Epoch  30: lam=0.217  EO=0.1368 [f=0.13]  acc=0.6737
    Epoch  35: lam=0.237  EO=0.0463 [f=0.04]  acc=0.6632
    Epoch  40: lam=0.256  EO=0.0671 [f=0.06]  acc=0.6470
    Epoch  45: lam=0.27

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'score_text': 'N', 'target': 'N', 'c_charge_desc': 'N', 'sex': 'N', 'decile_score': 'N', 'race': 'N', 'nn_conf': 'N', 'nn_pred': 'N', 'priors_count': 'N'}


    BBN modified 165/1235 (13.4%)
  Stage 3: Additive group-threshold calibration...
    Threshold floor: 0.6780 (ref=0.6980, drop=0.020)
    Grid best: EO=0.0282 [floor=0.02]  acc=0.6785  thresholds={'race': (np.float64(0.36), np.float64(0.39)), 'sex': (np.float64(0.64), np.float64(0.63))}
      race: TPR=(0.600,0.599) FPR=(0.268,0.240) EO=0.0282 [floor=0.02]
      sex: TPR=(0.604,0.576) FPR=(0.253,0.264) EO=0.0273 [floor=0.02]

COMPAS RESULTS
Accuracy: baseline=0.6980  fair=0.6785  drop=0.0194 (1.94%)  [ACCEPTABLE]
EO Metrics:
  race          : 0.26 → 0.02  [TARGET ✓]
  sex           : 0.23 → 0.02  [TARGET ✓]

GERMAN
Loading cached German data...
Training baseline MLP...
  MLP baseline acc: 0.7250

Training adversarial adapter...
  Stage 1: Task + group-balance + positive alignment...
    Early stop at epoch 145
    Stage 1 acc=0.7150  EO=0.3558 [floor=0.35]
  Stage 2 [PATH A — GRL+MMD+EO]: lambda 0.05→0.4  mmd_w=1.2  direct_eo_w=4.0  enc_lr=1.25e-04  adv_lr_factor=0.5
    Epoch   0:

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'target': 'N', 'sex_label': 'N', 'housing': 'N', 'nn_conf': 'N', 'status': 'N', 'amount': 'N', 'nn_pred': 'N'}


    Adversary acc: 0.859 [adversary dominant ✗]
    Stage 2 EO=0.1941 [floor=0.19]  acc=0.7050
  BBN calibration (weight=0.2, threshold=0.4, fairness_dir=True)...
    BBN modified 66/200 (33.0%)
  Stage 3: Additive group-threshold calibration...
    Threshold floor: 0.7050 (ref=0.7250, drop=0.020)
    Grid best: EO=0.0222 [floor=0.02]  acc=0.7150  thresholds={'age': (np.float64(0.36), np.float64(0.32)), 'sex': (np.float64(0.48), np.float64(0.6))}
      age: TPR=(0.688,0.710) FPR=(0.250,0.271) EO=0.0222 [floor=0.02]
      sex: TPR=(0.709,0.692) FPR=(0.264,0.286) EO=0.0216 [floor=0.02]

GERMAN RESULTS
Accuracy: baseline=0.7250  fair=0.7150  drop=0.0100 (1.00%)  [GOOD]
EO Metrics:
  age           : 0.11 → 0.02  [TARGET ✓]
  sex           : 0.32 → 0.02  [TARGET ✓]

BANK
Loading cached Bank data...
Training baseline MLP...
  MLP baseline acc: 0.8451

Training adversarial adapter...
  Stage 1: Task + group-balance + positive alignment...
    Stage 1 acc=0.8368  EO=0.1261 [floor=0.12]
  Stage

...

    BBN modified 64/1569 (4.1%)
  Stage 3: Additive group-threshold calibration...
    Threshold floor: 0.8251 (ref=0.8451, drop=0.020)
    Grid best: EO=0.0171 [floor=0.01]  acc=0.8260  thresholds={'marital': (np.float64(0.53), np.float64(0.49)), 'job': (np.float64(1.09), np.float64(1.17))}
      marital: TPR=(0.535,0.520) FPR=(0.094,0.080) EO=0.0150 [floor=0.01]
      job: TPR=(0.528,0.522) FPR=(0.082,0.099) EO=0.0171 [floor=0.01]

BANK RESULTS
Accuracy: baseline=0.8451  fair=0.8260  drop=0.0191 (1.91%)  [ACCEPTABLE]
EO Metrics:
  job           : 0.08 → 0.01  [TARGET ✓]
  marital       : 0.05 → 0.01  [TARGET ✓]

LAWSCHOOL
Loading cached Law School data...
Training baseline MLP...
  MLP baseline acc: 0.9495

Training adversarial adapter...
  Stage 1: Task + group-balance + positive alignment...
    Stage 1 acc=0.8548  EO=0.2109 [floor=0.21]
  Stage 2 [PATH B — frozen encoder]: lambda 0.05→0.3  tau=0.5
    Epoch   0: lam=0.050  EO=0.2465 [f=0.24]  acc=0.7756
    Epoch   5: lam=0.066  E

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'target': 'N', 'bar': 'N', 'race': 'N', 'nn_conf': 'N', 'bar2': 'N', 'decile3': 'N', 'nn_pred': 'N', 'bar1': 'N', 'gender': 'N'}


    BBN modified 489/4478 (10.9%)
  Stage 3: Additive group-threshold calibration...
    Threshold floor: 0.9395 (ref=0.9495, drop=0.010)
    Note: entry acc=0.9393 < floor=0.9395. Searching for candidates that recover it.
    Grid best: EO=0.0000 [floor=0.00]  acc=0.9480  thresholds={'race': (np.float64(-0.2), np.float64(-0.2)), 'gender': (np.float64(-0.2), np.float64(-0.2))}
      race: TPR=(1.000,1.000) FPR=(1.000,1.000) EO=0.0000 [floor=0.00]
      gender: TPR=(1.000,1.000) FPR=(1.000,1.000) EO=0.0000 [floor=0.00]

LAWSCHOOL RESULTS
Accuracy: baseline=0.9495  fair=0.9480  drop=0.0016 (0.16%)  [EXCELLENT]
EO Metrics:
  gender        : 0.02 → 0.00  [TARGET ✓]
  race          : 0.23 → 0.00  [TARGET ✓]

HOSPITAL
Cached Hospital data to ./cache/preproc_hospital.pkl
Training baseline MLP...
  MLP baseline acc: 0.6298

Training adversarial adapter...
  Stage 1: Task + group-balance + positive alignment...
    Stage 1 acc=0.6244  EO=0.0560 [floor=0.05]
  Stage 2 [PATH A — GRL+MMD+EO]: lamb

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'target': 'N', 'num_medications': 'N', 'race': 'N', 'nn_conf': 'N', 'diabetesMed': 'N', 'nn_pred': 'N', 'number_diagnoses': 'N', 'age': 'N'}


    BBN modified 2735/10364 (26.4%)
  Stage 3: Additive group-threshold calibration...
    Threshold floor: 0.6098 (ref=0.6298, drop=0.020)
    Grid best: EO=0.0169 [floor=0.01]  acc=0.6120  thresholds={'race': (np.float64(0.29), np.float64(0.27)), 'sex': (np.float64(0.91), np.float64(0.93))}
      race: TPR=(0.301,0.289) FPR=(0.122,0.139) EO=0.0169 [floor=0.01]
      sex: TPR=(0.288,0.295) FPR=(0.138,0.131) EO=0.0077 [floor=0.00]

HOSPITAL RESULTS
Accuracy: baseline=0.6298  fair=0.6120  drop=0.0178 (1.78%)  [ACCEPTABLE]
EO Metrics:
  race          : 0.06 → 0.01  [TARGET ✓]
  sex           : 0.02 → 0.00  [TARGET ✓]

FINAL SUMMARY
COMPAS      : acc=0.6785 (drop=+0.0194)  EO=0.02  [SUCCESS ✓]
GERMAN      : acc=0.7150 (drop=+0.0100)  EO=0.02  [SUCCESS ✓]
BANK        : acc=0.8260 (drop=+0.0191)  EO=0.01  [SUCCESS ✓]
LAWSCHOOL   : acc=0.9480 (drop=+0.0016)  EO=0.00  [SUCCESS ✓]
HOSPITAL    : acc=0.6120 (drop=+0.0178)  EO=0.01  [SUCCESS ✓]
